In [ ]:
# 1. Install arc-agi runtime from offline competition wheelhouse
import os, sys, glob, subprocess

print("Installing arc-agi runtime from offline wheelhouse...")
wheel_dirs = glob.glob("/kaggle/input/**/arc_agi_3_wheels", recursive=True)
if wheel_dirs:
    wheel_dir = wheel_dirs[0]
    print(f"Found wheels directory: {wheel_dir}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheel_dir, "arc-agi", "python-dotenv", "pillow"], check=False)
else:
    print("arc_agi_3_wheels not found in /kaggle/input; dependencies should be pre-installed.")


In [ ]:
# 2. Cohezion ARC-AGI-3 SOTA v36: Monadic Markov + Soft-Affordance Guided BFS
# Lineage: v27 (0.13) -> v32 (0.16) -> v33 (0.21) -> v34 (0.15 trap) -> v35 Monotonic Recovery
import hashlib
import json
import logging
import os
import random
import sys
import time
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set
import numpy as np

import arc_agi
import arcengine
from arcengine import GameAction, GameState, FrameDataRaw

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("CohezionARC3")

# Precomputed 64-bit SplitMix64 Zobrist table for 64x64 grids across 10 discrete colors
RNG = np.random.default_rng(20261012)
ZOBRIST_TABLE = RNG.integers(0, (1 << 62) - 1, size=(64, 64, 10), dtype=np.int64)

OPPOSITE_ACTION = {1: 2, 2: 1, 3: 4, 4: 3}  # Up <-> Down, Left <-> Right
MOVES_4WAY = [(-1, 0, 1), (1, 0, 2), (0, -1, 3), (0, 1, 4)] # dr, dc, action_id

def hash_frame(frame_layers: list) -> int:
    """Vectorized 64-bit Zobrist XOR hash for O(1) grid state comparison."""
    if not frame_layers or frame_layers[0] is None:
        return 0
    grid = frame_layers[0]
    h, w = grid.shape
    if h > 64 or w > 64:
        return int(hashlib.md5(np.ascontiguousarray(grid).tobytes()).hexdigest()[:15], 16)
    colors = np.clip(grid, 0, 9)
    r_idx = np.arange(h)[:, None]
    c_idx = np.arange(w)
    return int(np.bitwise_xor.reduce(ZOBRIST_TABLE[:h, :w, :][r_idx, c_idx, colors].ravel()))

def find_components(grid: np.ndarray) -> tuple[list[dict[str, Any]], int]:
    """Extract foreground connected components sorted by color rarity ascending, then size ascending."""
    h, w = grid.shape
    visited = np.zeros((h, w), dtype=bool)
    components = []
    vals, counts = np.unique(grid, return_counts=True)
    bg_color = int(vals[np.argmax(counts)])
    color_freq = dict(zip(vals, counts))

    for r in range(h):
        for c in range(w):
            if visited[r, c] or grid[r, c] == bg_color:
                continue
            color = int(grid[r, c])
            q = deque([(r, c)])
            visited[r, c] = True
            pixels = []
            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))
                for dr, dc, _ in MOVES_4WAY:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == color:
                        visited[nr, nc] = True
                        q.append((nr, nc))

            if pixels:
                pr = [p[0] for p in pixels]
                pc = [p[1] for p in pixels]
                components.append({
                    "color": color,
                    "size": len(pixels),
                    "rarity": color_freq[color],
                    "center_r": int(np.mean(pr)),
                    "center_c": int(np.mean(pc)),
                    "bbox": (min(pr), min(pc), max(pr), max(pc)),
                    "pixels": pixels,
                })

    components.sort(key=lambda x: (x["rarity"], x["size"]))
    return components, bg_color

def soft_bfs_suggested_action(grid: np.ndarray, start: tuple[int, int], goal: tuple[int, int], bg_color: int) -> Optional[int]:
    """Soft-affordance BFS: provides single-step directional advice without locking the agent into blind execution."""
    h, w = grid.shape
    if start == goal:
        return None
    # Priority queue simulation via double-ended queue for 0-1 BFS
    # Edge cost: 1 if target or background color, 2 if non-background object (soft obstacle/item)
    dist = {start: 0}
    parent = {start: None}
    action_taken = {}
    q = deque([start])

    max_evals = 600
    evals = 0

    while q and evals < max_evals:
        evals += 1
        r, c = q.popleft()
        if (r, c) == goal:
            break
        for dr, dc, act in MOVES_4WAY:
            nr, nc = r + dr, c + dc
            if 0 <= nr < h and 0 <= nc < w:
                step_cost = 1 if int(grid[nr, nc]) in (bg_color, 0) or (nr, nc) == goal else 2
                new_dist = dist[(r, c)] + step_cost
                if (nr, nc) not in dist or new_dist < dist[(nr, nc)]:
                    dist[(nr, nc)] = new_dist
                    parent[(nr, nc)] = (r, c)
                    action_taken[(nr, nc)] = act
                    if step_cost == 1:
                        q.appendleft((nr, nc))
                    else:
                        q.append((nr, nc))

    if goal not in parent:
        return None

    # Trace back to find first step
    curr = goal
    while parent[curr] is not None and parent[curr] != start:
        curr = parent[curr]
    return action_taken.get(curr)

class CohezionMonadicMarkovAffordanceAgent:
    """SOTA v35: Monadic Markov Policy with Soft-Affordance Guided Navigation."""
    def __init__(self, game_id: str, seed: int = 42):
        self.game_id = game_id
        self.rng = random.Random(seed + hash(game_id) % 100000)
        self.transition_model: dict[int, dict[int, int]] = {}
        self.state_visits: dict[int, int] = {}
        self.state_action_visits: dict[tuple[int, int], int] = {}
        self.clicked_targets: dict[int, set[tuple[int, int]]] = {}
        self.last_state_hash: Optional[int] = None
        self.last_action_id: Optional[int] = None
        self.last_frame: Optional[np.ndarray] = None
        self.player_pos: Optional[tuple[int, int]] = None
        self.step_count = 0
        self.consecutive_loops = 0
        self.state_history: deque[int] = deque(maxlen=24)
        self.recent_hashes: deque[int] = deque(maxlen=8)

    def choose_action(self, obs: FrameDataRaw) -> tuple[GameAction, Optional[dict[str, Any]]]:
        self.step_count += 1
        curr_hash = hash_frame(obs.frame)
        self.state_visits[curr_hash] = self.state_visits.get(curr_hash, 0) + 1
        grid = obs.frame[0] if obs.frame else None

        # Track avatar position from frame delta on directional movement
        if self.last_frame is not None and grid is not None and self.last_action_id in (1, 2, 3, 4):
            diff = (grid != self.last_frame)
            if 0 < np.sum(diff) <= 64:
                coords = np.argwhere(diff)
                self.player_pos = (int(np.mean(coords[:, 0])), int(np.mean(coords[:, 1])))

        self.last_frame = grid.copy() if grid is not None else None

        # Track state transitions and consecutive self-loops (wall collisions & orbit traps)
        is_cycle = False
        if self.last_state_hash is not None and self.last_action_id is not None:
            if self.last_state_hash not in self.transition_model:
                self.transition_model[self.last_state_hash] = {}
            self.transition_model[self.last_state_hash][self.last_action_id] = curr_hash
            sa_pair = (self.last_state_hash, self.last_action_id)
            self.state_action_visits[sa_pair] = self.state_action_visits.get(sa_pair, 0) + 1

            if curr_hash == self.last_state_hash:
                self.consecutive_loops += 1
                is_cycle = True
            elif len(self.state_history) >= 2 and curr_hash == self.state_history[-2]:
                self.consecutive_loops += 1
                is_cycle = True
            elif curr_hash in self.recent_hashes:
                self.consecutive_loops += 1
                is_cycle = True
            else:
                self.consecutive_loops = 0

        self.state_history.append(curr_hash)
        self.recent_hashes.append(curr_hash)

        avail = obs.available_actions
        simple_ids = [a for a in avail if a not in (0, 6, 7)]
        has_click = 6 in avail

        if curr_hash not in self.clicked_targets:
            self.clicked_targets[curr_hash] = set()

        comps, bg_color = find_components(grid) if grid is not None else ([], 0)

        # 1. Click handling: Trigger on unclicked affordances, or breakout if trapped in loops
        if has_click and (not simple_ids or self.consecutive_loops >= 2 or self.rng.random() < 0.30):
            unclicked = [c for c in comps if (c["center_c"], c["center_r"]) not in self.clicked_targets[curr_hash]]
            if unclicked:
                target = unclicked[0]
                coord = (target["center_c"], target["center_r"])
                self.clicked_targets[curr_hash].add(coord)
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}
            elif comps and not simple_ids:
                target = self.rng.choice(comps)
                r_min, c_min, r_max, c_max = target["bbox"]
                coord = (self.rng.randint(c_min, c_max), self.rng.randint(r_min, r_max))
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}

        # 2. Directed Movement with Monadic Markov Cycle Evasion & Soft-Affordance BFS Guidance
        if simple_ids:
            known = self.transition_model.get(curr_hash, {})
            preferred_scores = {a: 0.0 for a in simple_ids}

            # Component vector alignment
            if self.player_pos is not None and comps:
                pr, pc = self.player_pos
                target = comps[0]
                tr, tc = target["center_r"], target["center_c"]
                dr, dc = tr - pr, tc - pc
                if abs(dr) > abs(dc):
                    preferred = [1 if dr < 0 else 2, 3 if dc < 0 else 4]
                else:
                    preferred = [3 if dc < 0 else 4, 1 if dr < 0 else 2]
                for idx, act_id in enumerate(preferred):
                    if act_id in preferred_scores:
                        preferred_scores[act_id] -= (10.0 - idx * 4.0)

                # Soft-Affordance BFS advice bonus
                if grid is not None:
                    suggested_act = soft_bfs_suggested_action(grid, (pr, pc), (tr, tc), bg_color)
                    if suggested_act and suggested_act in preferred_scores:
                        preferred_scores[suggested_act] -= 15.0

                # Action 5 (Interact/Push) proximity bonus when adjacent to affordance component
                if 5 in preferred_scores and (abs(dr) + abs(dc)) <= 1:
                    preferred_scores[5] -= 16.0

            action_penalties = {}
            untried = [a for a in simple_ids if a not in known]
            for act_id in simple_ids:
                penalty = preferred_scores[act_id]

                # Strongly penalize reversing back into immediate prior position if cycling
                if is_cycle and self.last_action_id in OPPOSITE_ACTION:
                    if act_id == OPPOSITE_ACTION[self.last_action_id]:
                        penalty += 50.0

                if act_id in known:
                    next_s = known[act_id]
                    # Self-loop penalty: Action confirmed to not move player (e.g. wall / immovable obstacle)
                    if next_s == curr_hash:
                        penalty += 100.0
                    # Recent-state orbit penalty: Prevents A -> B -> A -> B 2-cycles
                    if next_s in self.recent_hashes:
                        penalty += 35.0 * (self.recent_hashes.count(next_s) + 1)
                    # Global novelty penalty
                    penalty += 2.0 * self.state_visits.get(next_s, 0)
                else:
                    # Unexplored transition reward
                    penalty -= 5.0

                # Frequency of (s, a) execution penalty
                sa_count = self.state_action_visits.get((curr_hash, act_id), 0)
                penalty += 1.5 * sa_count

                # Orthogonal escape pulse on stubborn cycles
                if is_cycle and self.consecutive_loops >= 3 and act_id in untried:
                    penalty -= 80.0

                # Jitter to break ties deterministically
                penalty += self.rng.uniform(0.0, 0.5)
                action_penalties[act_id] = penalty

            best_action_id = min(action_penalties, key=action_penalties.get)
            self.last_state_hash = curr_hash
            self.last_action_id = best_action_id
            return GameAction.from_id(best_action_id), None

        # Ultimate fallback
        chosen = self.rng.choice(avail) if avail else 0
        self.last_state_hash = curr_hash
        self.last_action_id = chosen
        return GameAction.from_id(chosen), None

# Backward compatibility bindings
CohezionHierarchicalPlannerAgent = CohezionMonadicMarkovAffordanceAgent
CohezionRaritySearchAgent = CohezionMonadicMarkovAffordanceAgent
CohezionGoExploreAgent = CohezionMonadicMarkovAffordanceAgent
CohezionDirectedRarityAgent = CohezionMonadicMarkovAffordanceAgent
CohezionMonadicMarkovRarityAgent = CohezionMonadicMarkovAffordanceAgent


In [ ]:
# 3. Competition Arcade Master Orchestrator (SOTA v36: 75-Step Dynamic Horizon + Liveness Watchdog)
import os, sys, glob, time, urllib.request
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
print(f"Cohezion ARC-AGI-3 Master Orchestrator: TRUE_SUBMISSION={TRUE_SUBMISSION}")

GATEWAY_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001/")
MAX_TOTAL_RUNTIME_S = 8.5 * 3600
START_TIME = time.time()

def wait_for_gateway(base_url: str, max_wait_s: int = 300) -> bool:
    print(f"Polling gateway at {base_url}api/games (up to {max_wait_s}s)...", flush=True)
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        try:
            req = urllib.request.Request(f"{base_url}api/games")
            with urllib.request.urlopen(req, timeout=5) as resp:
                if resp.status == 200:
                    print("✓ Gateway is online and ready!", flush=True)
                    return True
        except Exception:
            pass
        time.sleep(3)
    print("Warning: Gateway polling timed out.", flush=True)
    return False

if TRUE_SUBMISSION:
    wait_for_gateway(GATEWAY_URL)
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=GATEWAY_URL,
        arc_api_key="test-key-123",
    )
else:
    env_dirs = glob.glob("/kaggle/input/**/environment_files", recursive=True)
    env_dir = env_dirs[0] if env_dirs else "data/arc_prize/environment_files"
    print(f"Running in OFFLINE mode using environments from: {env_dir}")
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )

card_id = arcade.create_scorecard()
print(f"Opened Scorecard: {card_id}")

try:
    envs = arcade.get_environments()
    print(f"Discovered {len(envs)} environments to solve.")

    # In offline commit mode, run 2 environments quickly to pass the commit gate
    if not TRUE_SUBMISSION:
        envs = envs[:2]

    for idx, env_info in enumerate(envs):
        elapsed = time.time() - START_TIME
        if elapsed > MAX_TOTAL_RUNTIME_S:
            print(f"Time budget reached ({elapsed:.1f}s). Finalizing scorecard.")
            break

        game_id = env_info.game_id
        print(f"[{idx+1}/{len(envs)}] Playing {game_id} (Elapsed: {elapsed:.1f}s)...", flush=True)
        try:
            env = arcade.make(game_id, scorecard_id=card_id)
            if env is None:
                continue
            agent = CohezionGoExploreAgent(game_id, seed=idx * 100)
            obs = env.reset()

            step = 0
            # SOTA v32 RHAE Action Invariant: Cap at 35 steps to preserve quadratic efficiency (N_human / N_AI)^2
            while obs and obs.state not in (GameState.WIN, GameState.GAME_OVER) and step < 75:
                # Liveness Watchdog: Breakout if stuck in 8 consecutive cycle traps (wall collision / orbit)
                if getattr(agent, "consecutive_loops", 0) >= 8:
                    print(f"  Watchdog: 8 consecutive cycle traps on {game_id} at step {step}. Early terminating to preserve RHAE.")
                    break
                act, data = agent.choose_action(obs)
                obs = env.step(act, data=data)
                step += 1

            print(f"  -> {game_id} finished in {step} steps. State: {obs.state if obs else None}, Levels: {obs.levels_completed if obs else 0}")
        except Exception as exc:
            print(f"  Error on {game_id}: {exc}")
finally:
    sc = arcade.close_scorecard(card_id)
    print(f"✓ Closed Scorecard {card_id}. Total actions: {sc.total_actions if sc else 0}, Completed: {sc.total_levels_completed if sc else 0}")


In [ ]:
# 4. Commit Validator Parquet Handler
# During commit (not TRUE_SUBMISSION), generates initial valid parquet so Kaggle enables Submit.
# During competition rerun (TRUE_SUBMISSION), Kaggle generates submission.parquet automatically from the scorecard.
import os
import pandas as pd
from pathlib import Path

target_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
target_file = target_dir / "submission.parquet"

if not TRUE_SUBMISSION:
    print("Generating commit-time validator submission.parquet...")
    commit_df = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"]
    )
    commit_df.to_parquet(target_file, index=False)
    print(f"✓ Commit validator submission.parquet written ({target_file.stat().st_size} bytes).")
else:
    print("TRUE_SUBMISSION rerun complete. Gateway recorded scorecard for evaluation.")
